## TP — De l'arbre à la forêt : Anticiper les interventions d'urgence

Une direction du Ministère de la Transition écologique souhaite mieux anticiper les interventions de maintenance sur un ensemble d'infrastructures publiques (ponts, routes, tunnels).

L'objectif de ce TP est de construire un modèle capable de répondre à une question simple : **Une intervention urgente sera-t-elle nécessaire prochainement ?**

### Déroulement

| Niveau       | Activité                                                     |
| ------------ | ------------------------------------------------------------ |
| **Niveau 1** | Construire et comprendre une Random Forest                   |
| **Niveau 2** | Comprendre l'effet du nombre d'arbres et de la randomisation |
| **Niveau 3** | Challenge : améliorer, analyser et décider                   |



## Création du jeu de données

Exécutez le code suivant pour générer les données de notre parc d'infrastructures.

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_samples = 1000

X = pd.DataFrame({
    'infrastructure_age': rng.integers(1, 60, n_samples),
    'annual_incidents': rng.poisson(2, n_samples),
    'maintenance_delay': rng.integers(0, 120, n_samples),
    'weather_exposure': rng.uniform(0, 1, n_samples),
    'traffic_level': rng.integers(500, 15000, n_samples),
    'infrastructure_type': rng.choice(['bridge', 'road', 'tunnel'], n_samples),
    'region': rng.choice(['north', 'south', 'east', 'west'], n_samples)
})

# Génération de la cible avec une règle métier complexe et un peu de bruit
risk_score = (
    (X['infrastructure_age'] / 60) * 0.3 +
    (X['annual_incidents'] / 5) * 0.3 +
    (X['maintenance_delay'] / 120) * 0.2 +
    (X['weather_exposure']) * 0.2
)
# Ajout d'une part de hasard
risk_score += rng.normal(0, 0.1, n_samples)
y = (risk_score > 0.6).astype(int)


## Niveau 1 — Guidé : De l'arbre à la forêt

### Partie 1 — Découverte des données

Votre première mission est de comprendre le problème avant de modéliser.

* Affichez les premières lignes du dataset.
* Identifiez les variables numériques et catégorielles.
* Analysez la distribution de la cible `y` et vérifiez les valeurs manquantes.

**Questions :**

1. Quelle est la variable cible ?
2. Les classes à prédire sont-elles équilibrées ?

### Partie 2 — Construire un premier arbre

Séparez vos données et configurez votre preprocessor :

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# À vous de définir 'preprocessor' avec ColumnTransformer
# ...

decision_tree = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=None, random_state=42))
])

decision_tree.fit(x_train, y_train)
print(f"Train : {decision_tree.score(x_train, y_train):.3f}")
print(f"Test  : {decision_tree.score(x_test, y_test):.3f}")


**Question :** Pourquoi les performances sur les données d'entraînement sont-elles très supérieures à celles sur les données de test ?

### Partie 3 — Construire une Random Forest

Remplacez le modèle par une forêt aléatoire :

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

random_forest.fit(x_train, y_train)


**Mission :** Construisez un tableau comparant le score d'entraînement et de test entre le Decision Tree et la Random Forest. Que constatez-vous ?

### Partie 4 — Observer les arbres de la forêt

Ouvrons la "boîte noire" en examinant les arbres individuels générés.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

forest = random_forest.named_steps['model']
print(f"Nombre d'arbres : {len(forest.estimators_)}")

# Affichage du premier arbre
plt.figure(figsize=(20, 10))
plot_tree(forest.estimators_[0], max_depth=2, filled=True)
plt.title("Arbre 1")
plt.show()

# Affichage du second arbre (à vous de coder)
# ...


**Questions :**

1. Les deux arbres sont-ils identiques ?
2. Sachant qu'ils proviennent du même dataset initial, pourquoi sont-ils différents ?

### Partie 5 — Faire voter la forêt

Conceptuellement, chaque arbre émet une prédiction ("Urgent" ou "Non urgent").

In [ ]:
# Prenons la première infrastructure du jeu de test
sample = x_test.iloc[[0]]
sample_preprocessed = random_forest.named_steps['preprocessor'].transform(sample)

predictions_arbres = []
for tree in forest.estimators_:
    pred = tree.predict(sample_preprocessed)
    predictions_arbres.append(pred[0])

print(predictions_arbres[:10]) # Affichons les 10 premiers votes


**Question :** Comment passer de cette liste de prédictions individuelles à la prédiction finale de la forêt (`random_forest.predict(sample)`) ?



## Niveau 2 — Réflexion : Analyser le comportement

### Partie 6 — Combien d'arbres faut-il ?

Entraînez plusieurs forêts en faisant varier `n_estimators`.

In [ ]:
n_estimators_values = [1, 5, 10, 25, 50, 100, 200, 500]
# Stockez et tracez le score_test pour chaque valeur


**Questions :**

1. Que se passe-t-il après 50 arbres ?
2. Pourquoi n'est-il pas nécessaire de paramétrer 10 000 arbres ?

### Partie 7 — Le rôle de max_features

Le paramètre `max_features` contrôle combien de variables l'arbre peut observer à chaque séparation.
Testez et comparez les performances de la forêt avec :

* `max_features='sqrt'`
* `max_features='log2'`
* `max_features=None` (utilise toutes les variables à chaque nœud)

**Question de réflexion :** Pourquoi est-il fondamental de *limiter* le nombre de variables accessibles à chaque séparation pour créer une bonne forêt ?

### Partie 8 — La forêt contre l'arbre

Testez plusieurs profondeurs (`max_depth` = 2, 4, 6, 10, None) sur un Decision Tree simple ET sur une Random Forest.
Tracez un graphique comparatif des scores d'entraînement et de test.



## Niveau 3 — Challenge : Déploiement Métier

### Challenge 1 — Trouver la meilleure forêt

Construisez le meilleur modèle possible. Vous pouvez modifier `n_estimators`, `max_depth`, `min_samples_split`, `max_features`.
**Contrainte stricte :** Tout doit être packagé dans une `Pipeline`. Aucun preprocessing manuel n'est autorisé en dehors.

### Challenge 2 — Ne pas se limiter à l'accuracy

Une *accuracy* élevée suffit-elle ?

1. Importez `precision_score`, `recall_score`, `f1_score`, et `confusion_matrix`.
2. Affichez la matrice de confusion de votre meilleur modèle.

### Challenge 3 — Quelle variable est importante ?

Affichez l'attribut `feature_importances_` de votre modèle final (faites correspondre les scores aux noms de colonnes issus du preprocessor).
**Question :** Les variables jugées importantes par le modèle correspondent-elles à la logique métier ayant servi à générer les données ?

### Challenge 4 — Une décision métier

La direction vous demande : *« Peut-on utiliser ce modèle en production pour déclencher automatiquement des travaux de maintenance lourde sans intervention humaine ? »*

Rédigez une réponse argumentée. Intégrez dans votre réflexion :

* Le coût d'un "Faux Positif" (travaux inutiles).
* Le coût d'un "Faux Négatif" (effondrement d'un pont).
* Le besoin de validation métier face à la "boîte noire".



## Activité Bonus — Cassons notre forêt !

Une forêt n'est pas juste "beaucoup d'arbres". Entraînez une **Forêt B** avec `n_estimators=100` mais désactivez la diversité (par exemple avec `bootstrap=False` et `max_features=None`).
Comparez ses performances avec la Forêt A du Niveau 1. Que se passe-t-il lorsque tous les arbres deviennent identiques ?